# 3.4 — Fase 4: Teste de Equivalência Estatística (H1)

Para cada (arquitetura, output, k ∈ {4,5,6}), comparar R²_baseline(k=8) vs. R²_reduzido(k) no test set via bootstrap pareado.

**Hipótese H1:** existe k* ∈ {4,5,6} tal que, para os três outputs, IC 95% bootstrap de ΔR² = R²_8 − R²_k satisfaz IC_sup ≤ δ = 0.02.

**Decisões:** D-E3-08 (δ=0.02), D-E3-09 (bootstrap pareado).

## Seção 1 — Imports e configuração

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mlflow

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR       = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
PROCESSED_DIR  = BASE_DIR / "ARTEFATOS" / "ETAPA_0" / "processed"
ETAPA2_DIR     = BASE_DIR / "ARTEFATOS" / "ETAPA_2"
ETAPA3_DIR     = BASE_DIR / "ARTEFATOS" / "ETAPA_3"
SELECAO_DIR    = ETAPA3_DIR / "selecao"
REDUZIDO_DIR   = ETAPA3_DIR / "reduzido"
EQUIV_DIR      = ETAPA3_DIR / "equivalencia"
EQUIV_DIR.mkdir(exist_ok=True)

MLFLOW_URI = f"file:///{BASE_DIR}/mlruns"
mlflow.set_tracking_uri(MLFLOW_URI)

# Constantes do bootstrap
B     = 1000   # réplicas
DELTA = 0.02   # margem de equivalência (D-E3-08)
ALPHA = 0.05   # nível de significância → IC 95%

FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
FEAT_IDX      = {f: i for i, f in enumerate(FEATURE_NAMES)}
OUTPUT_NAMES  = ["M_CH3OH", "x_CH3OH", "ET"]
OUTPUT_IDX    = {o: i for i, o in enumerate(OUTPUT_NAMES)}
SKLEARN_MODELS = ["SVR", "DT", "RF", "XGBoost"]

print("Configuração OK")
print(f"  BASE_DIR:      {BASE_DIR}")
print(f"  EQUIV_DIR:     {EQUIV_DIR}")
print(f"  Bootstrap B={B}, δ={DELTA}")

## Seção 2 — Carga dos dados, subconjuntos e modelos

In [ ]:
# --- dados ---
X_test         = np.load(PROCESSED_DIR / "X_test.npy")
y_test         = np.load(PROCESSED_DIR / "y_test.npy")
scaler_y_min   = np.load(PROCESSED_DIR / "scaler_y_min.npy")
scaler_y_scale = np.load(PROCESSED_DIR / "scaler_y_scale.npy")

N_TEST = X_test.shape[0]
print(f"X_test: {X_test.shape}  y_test: {y_test.shape}  (N_TEST={N_TEST})")

# --- subconjuntos S_k ---
subsets_df = pd.read_csv(SELECAO_DIR / "3.2_subconjuntos.csv")
subsets    = {}
for _, row in subsets_df.iterrows():
    k     = int(row["k"])
    feats = [f.strip() for f in row["features"].split(",")]
    subsets[k] = feats

for k, feats in sorted(subsets.items()):
    idx = [FEAT_IDX[f] for f in feats]
    print(f"  S_{k}: {feats}  →  cols {idx}")

In [ ]:
def denorm_y(y_norm, output_name):
    i = OUTPUT_IDX[output_name]
    return y_norm * scaler_y_scale[i] + scaler_y_min[i]


def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0


def load_baseline_sklearn(model_name, output_name):
    path = ETAPA2_DIR / model_name / output_name / "model.pkl"
    with open(path, "rb") as f:
        return pickle.load(f)


def load_baseline_ann(output_name):
    import tensorflow as tf
    path = ETAPA2_DIR / "ANN" / output_name / "model_v2.keras"
    return tf.keras.models.load_model(str(path), compile=False)


def load_reduzido_sklearn(model_name, output_name, k):
    path = REDUZIDO_DIR / model_name / output_name / f"k{k}" / "model.pkl"
    with open(path, "rb") as f:
        return pickle.load(f)


def load_reduzido_ann(output_name, k):
    import tensorflow as tf
    path = REDUZIDO_DIR / "ANN" / output_name / f"k{k}" / "model.keras"
    return tf.keras.models.load_model(str(path), compile=False)


print("Funções auxiliares definidas.")

In [ ]:
# Pré-carregar TF uma vez para evitar overhead por modelo
import tensorflow as tf
print(f"TensorFlow {tf.__version__}")

# predicoes[modelo][output][k] = array shape (N_TEST,) — valores DESNORMALIZADOS
predicoes = {m: {o: {} for o in OUTPUT_NAMES} for m in SKLEARN_MODELS + ["ANN"]}

# --- Baseline sklearn (k=8, todas as features) ---
print("\n=== Baseline sklearn ===")
for model_name in SKLEARN_MODELS:
    for output_name in OUTPUT_NAMES:
        m = load_baseline_sklearn(model_name, output_name)
        p_norm = m.predict(X_test)
        predicoes[model_name][output_name][8] = denorm_y(p_norm, output_name)
        y_true = denorm_y(y_test[:, OUTPUT_IDX[output_name]], output_name)
        r2 = r2_score_manual(y_true, predicoes[model_name][output_name][8])
        print(f"  {model_name:8s} {output_name:10s}  R²={r2:.4f}")

# --- Baseline ANN (k=8, model_v2.keras) ---
print("\n=== Baseline ANN ===")
for output_name in OUTPUT_NAMES:
    ann = load_baseline_ann(output_name)
    p_norm = ann.predict(X_test, verbose=0).flatten()
    predicoes["ANN"][output_name][8] = denorm_y(p_norm, output_name)
    y_true = denorm_y(y_test[:, OUTPUT_IDX[output_name]], output_name)
    r2 = r2_score_manual(y_true, predicoes["ANN"][output_name][8])
    print(f"  ANN      {output_name:10s}  R²={r2:.4f}")

print("\n=== Modelos reduzidos sklearn ===")
for k in [4, 5, 6]:
    col_idx = [FEAT_IDX[f] for f in subsets[k]]
    X_te_k  = X_test[:, col_idx]
    for model_name in SKLEARN_MODELS:
        for output_name in OUTPUT_NAMES:
            m = load_reduzido_sklearn(model_name, output_name, k)
            p_norm = m.predict(X_te_k)
            predicoes[model_name][output_name][k] = denorm_y(p_norm, output_name)
            y_true = denorm_y(y_test[:, OUTPUT_IDX[output_name]], output_name)
            r2 = r2_score_manual(y_true, predicoes[model_name][output_name][k])
            print(f"  k={k}  {model_name:8s} {output_name:10s}  R²={r2:.4f}")

print("\n=== Modelos reduzidos ANN ===")
for k in [4, 5, 6]:
    col_idx = [FEAT_IDX[f] for f in subsets[k]]
    X_te_k  = X_test[:, col_idx]
    for output_name in OUTPUT_NAMES:
        ann = load_reduzido_ann(output_name, k)
        p_norm = ann.predict(X_te_k, verbose=0).flatten()
        predicoes["ANN"][output_name][k] = denorm_y(p_norm, output_name)
        y_true = denorm_y(y_test[:, OUTPUT_IDX[output_name]], output_name)
        r2 = r2_score_manual(y_true, predicoes["ANN"][output_name][k])
        print(f"  k={k}  ANN      {output_name:10s}  R²={r2:.4f}")

In [ ]:
# Salvar matrizes de predição
save_dict = {}
for model_name in SKLEARN_MODELS + ["ANN"]:
    for output_name in OUTPUT_NAMES:
        for k_val, preds in predicoes[model_name][output_name].items():
            key = f"{model_name}__{output_name}__k{k_val}"
            save_dict[key] = preds

np.savez(EQUIV_DIR / "3.4_predicoes_test.npz", **save_dict)
print(f"Predições salvas: {EQUIV_DIR / '3.4_predicoes_test.npz'}  ({len(save_dict)} arrays)")

## Seção 3 — Bootstrap pareado (D-E3-09)

ΔR² = R²_8 − R²_k — positivo significa baseline melhor.
Equivalência se IC_sup ≤ δ = 0.02 (D-E3-08).

In [ ]:
np.random.seed(RANDOM_STATE)

boot_results = []

for model_name in SKLEARN_MODELS + ["ANN"]:
    for output_name in OUTPUT_NAMES:
        y_true = denorm_y(y_test[:, OUTPUT_IDX[output_name]], output_name)
        p8     = predicoes[model_name][output_name][8]
        r2_8   = r2_score_manual(y_true, p8)

        for k in [4, 5, 6]:
            pk = predicoes[model_name][output_name][k]
            r2_k = r2_score_manual(y_true, pk)

            delta_r2_boot = np.empty(B)
            for b in range(B):
                idx = np.random.choice(N_TEST, size=N_TEST, replace=True)
                yt_b  = y_true[idx]
                r2_8_b = r2_score_manual(yt_b, p8[idx])
                r2_k_b = r2_score_manual(yt_b, pk[idx])
                delta_r2_boot[b] = r2_8_b - r2_k_b

            ic_inf = np.percentile(delta_r2_boot, 100 * ALPHA / 2)
            ic_sup = np.percentile(delta_r2_boot, 100 * (1 - ALPHA / 2))
            delta_medio = delta_r2_boot.mean()
            equivalente = bool(ic_sup <= DELTA)

            boot_results.append({
                "arquitetura":  model_name,
                "output":       output_name,
                "k":            k,
                "r2_8":         r2_8,
                "r2_k":         r2_k,
                "delta_r2":     r2_8 - r2_k,
                "delta_r2_boot_medio": delta_medio,
                "ic_inf":       ic_inf,
                "ic_sup":       ic_sup,
                "equivalente":  equivalente,
            })

print(f"Bootstrap concluído: {len(boot_results)} testes (5 arquiteturas × 3 outputs × 3 k)")

## Seção 4 — Tabela de resultados

In [ ]:
results_df = pd.DataFrame(boot_results)

# Formatar para exibição
display_cols = ["arquitetura", "output", "k", "r2_8", "r2_k",
                "delta_r2", "ic_inf", "ic_sup", "equivalente"]
disp = results_df[display_cols].copy()
for col in ["r2_8", "r2_k", "delta_r2", "ic_inf", "ic_sup"]:
    disp[col] = disp[col].round(4)

print("=== Resultados do teste de equivalência ===")
try:
    from IPython.display import display
    display(disp.sort_values(["arquitetura", "output", "k"]).reset_index(drop=True))
except Exception:
    print(disp.sort_values(["arquitetura", "output", "k"]).reset_index(drop=True).to_string())

# Salvar CSV
results_df.to_csv(EQUIV_DIR / "3.4_equivalencia_resultados.csv", index=False)
print(f"\nSalvo: {EQUIV_DIR / '3.4_equivalencia_resultados.csv'}")
print(f"Total equivalente: {results_df['equivalente'].sum()} / {len(results_df)}")

In [ ]:
# Resumo: equivalência por (k, arquitetura) — quantos outputs simultâneos
pivot_equiv = results_df.groupby(["k", "arquitetura"])["equivalente"].sum().unstack("arquitetura")
print("=== Nº de outputs equivalentes por (k, arquitetura) [máx = 3] ===")
try:
    display(pivot_equiv)
except Exception:
    print(pivot_equiv.to_string())

# Candidatos a k* (equivalente nos 3 outputs)
full_equiv = results_df.groupby(["k", "arquitetura"])["equivalente"].sum()
candidatos = full_equiv[full_equiv == 3].reset_index()
print("\n=== Candidatos a k* (equivalente nos 3 outputs simultaneamente) ===")
if candidatos.empty:
    print("Nenhuma combinação (k, arquitetura) equivalente nos 3 outputs.")
    print("Cenário de fallback — ver 3.5.")
else:
    print(candidatos.to_string())

## Seção 5 — Visualizações

In [ ]:
# Forest plot: IC 95% de ΔR² por output (3 subplots)
MODELS_ORDER = ["SVR", "DT", "RF", "XGBoost", "ANN"]
K_COLORS     = {4: "#e15759", 5: "#f28e2b", 6: "#4e79a7"}
K_MARKERS    = {4: "o", 5: "s", 6: "^"}  # círculo, quadrado, triângulo

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax_idx, output_name in enumerate(OUTPUT_NAMES):
    ax = axes[ax_idx]
    sub = results_df[results_df["output"] == output_name].copy()

    yticks = []
    yticklabels = []
    y_pos = 0

    for model_name in MODELS_ORDER:
        for k in [4, 5, 6]:
            row = sub[(sub["arquitetura"] == model_name) & (sub["k"] == k)]
            if row.empty:
                continue
            r = row.iloc[0]
            color = K_COLORS[k]
            marker = K_MARKERS[k]

            # IC bar
            ax.plot([r["ic_inf"], r["ic_sup"]], [y_pos, y_pos],
                    color=color, lw=1.8, solid_capstyle="round")
            # ponto médio
            ax.scatter([r["delta_r2_boot_medio"]], [y_pos],
                       color=color, marker=marker, s=50, zorder=5)

            yticks.append(y_pos)
            yticklabels.append(f"{model_name} k={k}")
            y_pos += 1

        y_pos += 0.5  # espaço entre modelos

    ax.axvline(DELTA, color="black", lw=1, ls="--", label=f"δ={DELTA}")
    ax.axvline(0, color="gray", lw=0.7, ls=":")
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels, fontsize=8)
    ax.set_xlabel("ΔR² = R²₈ − R²ₖ", fontsize=9)
    ax.set_title(output_name, fontsize=11, fontweight="bold")
    ax.grid(axis="x", lw=0.4, alpha=0.5)

# Legenda
legend_elements = [
    mpatches.Patch(color=K_COLORS[k], label=f"k={k}")
    for k in [4, 5, 6]
]
legend_elements.append(
    plt.Line2D([0], [0], color="black", ls="--", lw=1, label=f"δ={DELTA}")
)
axes[-1].legend(handles=legend_elements, loc="lower right", fontsize=8)

fig.suptitle("IC 95% Bootstrap de ΔR² = R²₈ − R²ₖ por Output", fontsize=13, y=1.01)
plt.tight_layout()
forest_path = EQUIV_DIR / "3.4_forest_plot.png"
plt.savefig(forest_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo: {forest_path}")

In [ ]:
# Heatmap: arquitetura × k × output com cor = equivalente
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

CMAP_BIN = matplotlib.colors.ListedColormap(["#d9534f", "#5cb85c"])  # False=vermelho, True=verde

for ax_idx, output_name in enumerate(OUTPUT_NAMES):
    ax = axes[ax_idx]
    sub = results_df[results_df["output"] == output_name]

    grid = np.zeros((len(MODELS_ORDER), 3), dtype=float)
    for m_i, model_name in enumerate(MODELS_ORDER):
        for k_i, k in enumerate([4, 5, 6]):
            row = sub[(sub["arquitetura"] == model_name) & (sub["k"] == k)]
            if not row.empty:
                grid[m_i, k_i] = float(row.iloc[0]["equivalente"])

    im = ax.imshow(grid, cmap=CMAP_BIN, vmin=0, vmax=1, aspect="auto")

    # anotações: ic_sup
    for m_i, model_name in enumerate(MODELS_ORDER):
        for k_i, k in enumerate([4, 5, 6]):
            row = sub[(sub["arquitetura"] == model_name) & (sub["k"] == k)]
            if not row.empty:
                val = row.iloc[0]["ic_sup"]
                color = "white" if bool(row.iloc[0]["equivalente"]) else "white"
                ax.text(k_i, m_i, f"{val:.3f}", ha="center", va="center",
                        fontsize=8, color="white", fontweight="bold")

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["k=4", "k=5", "k=6"])
    ax.set_yticks(range(len(MODELS_ORDER)))
    ax.set_yticklabels(MODELS_ORDER)
    ax.set_title(output_name, fontsize=11, fontweight="bold")

fig.suptitle(f"Equivalência (IC₉₅ sup ≤ δ={DELTA}) — verde=SIM, vermelho=NÃO\n(valor = IC 97.5%)",
             fontsize=11, y=1.02)
plt.tight_layout()
heatmap_path = EQUIV_DIR / "3.4_heatmap_equivalencia.png"
plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo: {heatmap_path}")

## Seção 6 — Logging de tags no MLflow

Adiciona `equivalente_a_baseline`, `delta_r2_medio`, `ic_inf`, `ic_sup` às 45 runs do experimento `reduzido` (fase=retreino).

In [ ]:
df_reduzido = mlflow.search_runs(
    experiment_names=["reduzido"],
    filter_string="tags.fase = 'retreino'",
)
print(f"Runs retreino no experimento reduzido: {len(df_reduzido)}")

# Construir lookup: (arquitetura, output, k) → run_id
run_lookup = {}
for _, row in df_reduzido.iterrows():
    arq  = row.get("params.arquitetura", "")
    out  = row.get("params.output", "")
    k_v  = str(row.get("params.k", ""))
    if arq and out and k_v:
        run_lookup[(arq, out, int(k_v))] = row["run_id"]

print(f"Lookup construído: {len(run_lookup)} runs mapeadas")

tagged = 0
missing = []

for _, row in results_df.iterrows():
    key = (row["arquitetura"], row["output"], int(row["k"]))
    run_id = run_lookup.get(key)
    if run_id is None:
        missing.append(key)
        continue

    mlflow.tracking.MlflowClient().set_tag(run_id, "equivalente_a_baseline", str(row["equivalente"]))
    mlflow.tracking.MlflowClient().set_tag(run_id, "delta_r2_medio",         f"{row['delta_r2_boot_medio']:.6f}")
    mlflow.tracking.MlflowClient().set_tag(run_id, "ic_inf",                 f"{row['ic_inf']:.6f}")
    mlflow.tracking.MlflowClient().set_tag(run_id, "ic_sup",                 f"{row['ic_sup']:.6f}")
    tagged += 1

print(f"Tags adicionadas: {tagged} / {len(results_df)} runs")
if missing:
    print(f"AVISO — runs não encontradas no MLflow: {missing}")

## Seção 7 — Validação final

In [ ]:
# Verificações programáticas de acordo com 3_PLANO.md
assert len(results_df) == 45, f"Esperado 45 linhas, obtido {len(results_df)}"
assert results_df[["ic_inf", "ic_sup"]].notna().all().all(), "NaN encontrado nos ICs"
assert (EQUIV_DIR / "3.4_equivalencia_resultados.csv").exists()
assert (EQUIV_DIR / "3.4_forest_plot.png").exists()
assert (EQUIV_DIR / "3.4_heatmap_equivalencia.png").exists()
assert (EQUIV_DIR / "3.4_predicoes_test.npz").exists()

n_equiv = results_df['equivalente'].sum()
cands   = results_df.groupby(["k","arquitetura"])["equivalente"].sum()
k_star_cands = cands[cands == 3].reset_index()

print("Validação PASSOU")
print(f"  45 testes computados")
print(f"  {n_equiv}/45 equivalentes")
print(f"  Candidatos a k* (≡ nos 3 outputs): {len(k_star_cands)}")
if not k_star_cands.empty:
    print(k_star_cands.to_string())
print(f"  Tags adicionadas: {tagged}/45")
print(f"\nPróximo passo: 3.5_selecao_k_estrela.ipynb")